# SVC_Combined — 최종 학습 및 추론

[1차 최종 선정 Manifest](https://github.com/ryu980920/Machine_Learning/blob/main/07_fine_tuning/05_final_models/CURRENT_FINAL_MODEL_SELECTION.json) — 기존 로컬 검토 패키지의 `02_SVC/full_runs/C2_G2/model.joblib`.

현재 선정: C2_G2 (C=2, gamma scale multiplier=2); 2차 실험 제외. 학습과 추론을 별도로 실행할 수 있다. 기존 private kit에서 추출한 `final_benchmark.py`와 `sources/`, 학습 시 `development/` 필요. 기존 체크포인트를 추론하는 경우 학습하지 않는다. 재학습 결과가 기존 checkpoint와 바이트 단위 일치한다고 보장하지 않는다. 이 Notebook은 전체 데이터 재실행 전이며 학습 완료를 주장하지 않는다. Original Test·교수님 Test는 모델 선택에 사용하지 않는다.

In [ ]:
from pathlib import Path
import sys, hashlib, json
MODEL = 'SVC_Combined'
KIT_DIR = None  # private kit 추출 위치 (final_benchmark.py 및 sources/ 포함)
OUT = Path('new_final_runs').resolve()
TRAIN = False
INFER = False
CHECKPOINT = None  # 승인된 원본 model.joblib 또는 이번 재학습 파일
EXPECTED_SHA256 = 'f88be338da4065315d943ca4150b5a425c24adf63733fada7da212c837ae4398'
INPUT = None  # 28x28 uint8 grayscale NPZ, X key
PREDICTION_CSV = Path('new_predictions.csv')
fb = None
if TRAIN or INFER:
    assert KIT_DIR is not None, 'private kit 경로 지정 필요'
    kit = Path(KIT_DIR).resolve()
    assert hashlib.sha256((kit/'final_benchmark.py').read_bytes()).hexdigest() == '97b0e6e2fdccaeaeb08508780bd8111df6a8e88d1ad3d3298e5d7f95e0e12c9d', '원본 benchmark 코드 해시 불일치'
    sys.path.insert(0,str(kit))
    import final_benchmark as fb
    import sklearn
    assert sklearn.__version__=='1.8.0'
    print(json.dumps(fb.CONFIG['models'][MODEL],indent=2))


## 최종 학습 (선택)
학습 원본의 development-only bundle을 준비한다. 새로운 폴더에 기록한다.

In [ ]:
if TRAIN:
    # C2_G2: original first fine-tuning configuration, not the old C=1 baseline.
    import numpy as np, joblib, time
    from sklearn.svm import SVC
    from threadpoolctl import threadpool_limits
    from sklearn.metrics import accuracy_score
    target=OUT / MODEL; target.mkdir(parents=True, exist_ok=False)
    off, writers=fb.read_training()
    with threadpool_limits(limits=2):
        x,y,meta=fb.train_features(MODEL,off,writers)
        assert x.shape == (82080,1108) and len(y)==82080
        gamma=float(2.0/(x.shape[1]*x.var()))
        estimator=SVC(C=2.0,gamma=gamma,cache_size=200)
        t=time.perf_counter(); estimator.fit(x,y,sample_weight=np.ones(len(y))); fit_s=time.perf_counter()-t
        checkpoint=target/'model.joblib';joblib.dump(estimator,checkpoint,compress=3)
        fb.write_json(target/'fit_manifest.json',{'model':MODEL,'condition':'C2_G2','C':2.0,'gamma_multiplier':2.0,'gamma':gamma,'training_rows':82080,'seed':42,'fit_s':fit_s,'checkpoint_sha256':fb.digest(checkpoint),'original_test_used':False,'professor_test_used':False,'new_validation_used_in_fit':False,'augmentation':meta,'environment':fb.environment()})
else:
    print("Training skipped")

## 추론 (선택)
신뢰된 joblib만 로드한다. 새 출력 파일을 작성하며 기존 결과를 덮어쓰지 않는다.

In [ ]:
if INFER:
    import numpy as np, joblib, csv
    assert CHECKPOINT is not None and INPUT is not None and EXPECTED_SHA256, '원본 모델 SHA-256/입력 지정 필요'
    checkpoint=Path(CHECKPOINT).resolve()
    assert fb.digest(checkpoint)==EXPECTED_SHA256, '체크포인트 해시 불일치'
    with np.load(INPUT,allow_pickle=False) as z:
        raw=z['X']
    assert raw.dtype==np.uint8 and (raw.ndim==3 and raw.shape[1:]==(28,28) or raw.ndim==2 and raw.shape[1]==784)
    estimator=joblib.load(checkpoint)
    assert np.array_equal(estimator.classes_,np.arange(16))
    from threadpoolctl import threadpool_limits
    with threadpool_limits(limits=2):
        pred=estimator.predict(fb.feature(MODEL,raw.reshape(-1,784)))
    with PREDICTION_CSV.open('x',encoding='utf-8',newline='') as f:
        writer=csv.writer(f);writer.writerow(['index','prediction']);writer.writerows((i,fb.LABELS[int(v)]) for i,v in enumerate(pred))
    print(PREDICTION_CSV,len(pred))


## 기존 실측 근거
[10개 모델 평가](https://github.com/ryu980920/Machine_Learning/tree/main/06_final_validation) · [현재 최종 선정](https://github.com/ryu980920/Machine_Learning/blob/main/07_fine_tuning/05_final_models/CURRENT_FINAL_MODEL_SELECTION.json)